# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ndukwegodslove-del/FlyrankAI-MLinternship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
from huggingface_hub import whoami
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

print("Authenticated as:", whoami(token=HF_TOKEN)["name"])

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("DuckDB connection established.")

Authenticated as: Godslove888
DuckDB connection established.


In [2]:

REL = "hf://datasets/FlyRank/internship-warehouse"


TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Warehouse tables defined successfully!")

Warehouse tables defined successfully!


In [3]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("DuckDB connection restored!")

DuckDB connection restored!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        SUM(gsc_sum_position) * 1.0 /
            NULLIF(SUM(gsc_impressions), 0) AS march_avg_position,

        SUM(gsc_clicks) * 1.0 /
            NULLIF(SUM(gsc_impressions), 0) AS march_ctr,

        COUNT(DISTINCT report_date) AS march_days_observed

    FROM {TABLES['fact_daily']}

    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE

    GROUP BY client_hash_id, content_hash_id
""").df()

print("March feature records:", len(march_features))
display(march_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature records: 176738


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_ctr,march_days_observed
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.450877,0.001754,31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.298246,0.000000,26
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,5.637584,0.000000,30
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.906404,0.004222,31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,3.950542,0.005776,31


In [5]:
april_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        COUNT(DISTINCT report_date) AS april_days_observed

    FROM {TABLES['fact_daily']}

    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE

    GROUP BY client_hash_id, content_hash_id
""").df()

print("April outcome records:", len(april_outcomes))
display(april_outcomes.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April outcome records: 194760


,client_hash_id,content_hash_id,april_impressions,april_days_observed
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0,28
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0,14
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0,12
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0,29
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0,27


In [6]:
# Merge March features with April outcomes
evaluation = march_features.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Retain records with at least 25 observed days in both months
evaluation = evaluation[
    (evaluation["march_days_observed"] >= 25) &
    (evaluation["april_days_observed"] >= 25)
].copy()

# Calculate percentage decline in impressions
evaluation["impressions_decline_pct"] = (
    (evaluation["march_impressions"] - evaluation["april_impressions"])
    / evaluation["march_impressions"]
) * 100

# Create the outcome label
evaluation["is_declining_label"] = (
    evaluation["impressions_decline_pct"] > 20
).astype(int)

print("Evaluation cohort:", len(evaluation))

print("\nOutcome distribution:")
display(evaluation["is_declining_label"].value_counts())

print("\nSample records:")
display(evaluation.head())

Evaluation cohort: 81742

Outcome distribution:


,count
is_declining_label,
0,41446
1,40296



Sample records:


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_ctr,march_days_observed,april_impressions,april_days_observed,impressions_decline_pct,is_declining_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.450877,0.001754,31,1151.0,30,-0.964912,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.298246,0.000000,26,73.0,26,-28.070175,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,5.637584,0.000000,30,98.0,27,34.228188,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.906404,0.004222,31,2275.0,30,-60.098522,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,3.950542,0.005776,31,6266.0,30,-126.209386,0


In [7]:
import pandas as pd

print("Pandas imported successfully!")

Pandas imported successfully!


In [8]:
# Divide pages into four search-volume groups
evaluation["volume_bucket"] = pd.qcut(
    evaluation["march_impressions"],
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"]
)

# Calculate decline rates for each group
volume_signal_check = (
    evaluation.groupby("volume_bucket", observed=False)
    .agg(
        n=("is_declining_label", "size"),
        declining_pages=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean"),
        avg_impressions=("march_impressions", "mean")
    )
    .reset_index()
)

volume_signal_check["decline_rate_pct"] = (
    volume_signal_check["decline_rate"] * 100
).round(2)

display(volume_signal_check)

,volume_bucket,n,declining_pages,decline_rate,avg_impressions,decline_rate_pct
0,Low,20449,8505,0.415913,216.602572,41.59
1,Medium-Low,20423,11341,0.555305,670.836263,55.53
2,Medium-High,20438,10966,0.536550,1903.344701,53.65
3,High,20432,9484,0.464174,10228.562647,46.42


**Signal 1: Search Volume (Quick-Win Logic)**

**Verdict: MIXED**

The volume signal showed a non-linear relationship with subsequent impression decline. The Medium-Low bucket had the highest decline rate (55.53%), followed by Medium-High (53.65%). The High-volume bucket recorded 46.42%, while the Low-volume bucket recorded 41.59%.

This suggests that higher search volume alone does not consistently identify pages more likely to experience future declines. However, high-volume pages may still deserve attention because improvements or losses in visibility can affect a larger number of impressions. Therefore, volume is useful as an indication of potential impact, but insufficient as a standalone predictor of decline.

In [9]:
import numpy as np

print("NumPy imported successfully!")

NumPy imported successfully!


In [10]:
# Group pages by their average search position
evaluation["position_bucket"] = pd.cut(
    evaluation["march_avg_position"],
    bins=[0, 3, 6, 10, 20, float("inf")],
    labels=["1–3", ">3–6", ">6–10", ">10–20", ">20"]
)

# Calculate the median CTR within each position group
position_median_ctr = (
    evaluation.groupby("position_bucket", observed=False)["march_ctr"]
    .transform("median")
)

# Compare each page's CTR with the median for its position group
evaluation["ctr_position_group"] = np.where(
    evaluation["march_ctr"] < position_median_ctr,
    "Below position median",
    "At/above position median"
)

# Summarise subsequent decline rates
ctr_position_signal_check = (
    evaluation.groupby(
        ["position_bucket", "ctr_position_group"],
        observed=False
    )
    .agg(
        n=("is_declining_label", "size"),
        declining_pages=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean"),
        avg_ctr=("march_ctr", "mean")
    )
    .reset_index()
)

ctr_position_signal_check["decline_rate_pct"] = (
    ctr_position_signal_check["decline_rate"] * 100
).round(2)

ctr_position_signal_check["avg_ctr_pct"] = (
    ctr_position_signal_check["avg_ctr"] * 100
).round(2)

display(ctr_position_signal_check)

,position_bucket,ctr_position_group,n,declining_pages,decline_rate,avg_ctr,decline_rate_pct,avg_ctr_pct
0,1–3,At/above position median,3859,1773,0.459445,0.005742,45.94,0.57
1,1–3,Below position median,3855,2744,0.711803,0.000801,71.18,0.08
2,>3–6,At/above position median,10334,4142,0.400813,0.005766,40.08,0.58
3,>3–6,Below position median,10333,6240,0.603890,0.000838,60.39,0.08
4,>6–10,At/above position median,8869,3425,0.386177,0.005313,38.62,0.53
5,>6–10,Below position median,8865,4997,0.563677,0.000429,56.37,0.04
6,>10–20,At/above position median,8098,3665,0.452581,0.004557,45.26,0.46
7,>10–20,Below position median,8097,4320,0.533531,0.000145,53.35,0.01
8,>20,At/above position median,19432,8990,0.462639,0.001250,46.26,0.12
9,>20,Below position median,0,0,NaN,NaN,NaN,NaN


**Signal 2: CTR Relative to Search Position (CTR-Fix Logic)**

**Verdict: CONFIRMED**

The results showed a consistent relationship between below-median CTR and subsequent impression decline across comparable search-position groups. Among pages ranking in positions 1–3, the below-median CTR group recorded a decline rate of 71.18%, compared with 45.94% for the at-or-above-median group. Similar patterns appeared in positions >3–6 (60.39% versus 40.08%), >6–10 (56.37% versus 38.62%), and >10–20 (53.35% versus 45.26%).

These findings support the CTR-fix logic: pages attracting fewer clicks than comparable pages may deserve closer review. However, the relationship does not establish causation, and the comparison is limited by the use of aggregated page-level CTR and average position. The >20 position group had no below-median observations and therefore provided no comparison.

### Baseline Action-Scoring Rule

The baseline prioritises content pages with below-median click-through rates compared with other pages in similar average search-position groups. The priority score combines the relative CTR gap with the logarithm of March search impressions, allowing pages with greater potential visibility impact to receive higher priority.

**Reason code:** `LOW_CTR_FOR_POSITION`

**Action label:** `REVIEW_SEARCH_SNIPPET`

**Scoring principle:** Pages with below-median CTR receive a positive score based on their relative CTR gap and search visibility. Pages without this signal are not included in the action queue.

The rule uses March 2026 information only. April performance and the decline label are excluded from scoring.

In [11]:

# Calculate each page's CTR shortfall relative to its position group
evaluation["ctr_gap"] = (
    position_median_ctr - evaluation["march_ctr"]
)

# Calculate the relative CTR gap
evaluation["relative_ctr_gap"] = np.where(
    (position_median_ctr > 0) & (evaluation["ctr_gap"] > 0),
    evaluation["ctr_gap"] / position_median_ctr,
    0
)

# Calculate the baseline priority score
evaluation["priority_score"] = (
    evaluation["relative_ctr_gap"]
    * np.log1p(evaluation["march_impressions"])
)

# Keep only pages with a positive priority score
action_queue = evaluation[
    evaluation["priority_score"] > 0
].copy()

# Assign one reason code and one action label
action_queue["reason_code"] = "LOW_CTR_FOR_POSITION"
action_queue["action_label"] = "REVIEW_SEARCH_SNIPPET"

# Rank pages from highest to lowest priority
action_queue = action_queue.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

print("Scoring rule executed successfully.")
print("Total pages in evaluation cohort:", len(evaluation))
print("Pages selected for action:", len(action_queue))

display(
    action_queue[
        [
            "client_hash_id",
            "content_hash_id",
            "march_impressions",
            "march_ctr",
            "priority_score",
            "reason_code",
            "action_label"
        ]
    ].head(10)
)

Scoring rule executed successfully.
Total pages in evaluation cohort: 81742
Pages selected for action: 31150


,client_hash_id,content_hash_id,march_impressions,march_ctr,priority_score,reason_code,action_label
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,0.000007,11.772838,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,0.000008,11.685355,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
2,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,0.000113,11.631466,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834.0,0.000012,11.274672,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
4,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,0.000045,11.097423,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
5,client_73cda7b4e4f265ea,content_425715547c6a3ea8,71513.0,0.000042,10.899587,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
6,client_a80fca3f171ed1de,content_046fc480045b88f5,83788.0,0.000072,10.854679,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
7,client_a80fca3f171ed1de,content_f0703fc6ae385591,63201.0,0.000032,10.846655,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
8,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,0.000139,10.628108,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
9,client_62f4a7e64f5e0096,content_0c5606abaaab3178,38865.0,0.000000,10.567875,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET


In [12]:
# Validate the ranked action queue

print("Missing priority scores:",
      action_queue["priority_score"].isna().sum())

print("Infinite priority scores:",
      np.isinf(action_queue["priority_score"]).sum())

print("Non-positive priority scores:",
      (action_queue["priority_score"] <= 0).sum())

print("Duplicate client-content pairs:",
      action_queue.duplicated(
          subset=["client_hash_id", "content_hash_id"]
      ).sum())

print("Incorrect reason codes:",
      (action_queue["reason_code"] != "LOW_CTR_FOR_POSITION").sum())

print("Incorrect action labels:",
      (action_queue["action_label"] != "REVIEW_SEARCH_SNIPPET").sum())

print("Ranking is descending:",
      action_queue["priority_score"].is_monotonic_decreasing)

Missing priority scores: 0
Infinite priority scores: 0
Non-positive priority scores: 0
Duplicate client-content pairs: 0
Incorrect reason codes: 0
Incorrect action labels: 0
Ranking is descending: True


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
from pathlib import Path

# Create the output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output file
output_path = output_dir / "baseline_action_score.csv"

# Export the ranked queue
action_queue.to_csv(output_path, index=False)

print("CSV exported successfully!")
print("File location:", output_path)
print("Number of exported rows:", len(action_queue))
print("File size (KB):", round(output_path.stat().st_size / 1024, 2))

CSV exported successfully!
File location: work/outputs/baseline_action_score.csv
Number of exported rows: 31150
File size (KB): 7751.16


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
# Select the ten highest-priority pages for manual review
top_10_review = action_queue.head(10).copy()

# Display the information needed for review
display(
    top_10_review[
        [
            "client_hash_id",
            "content_hash_id",
            "position_bucket",
            "march_avg_position",
            "march_impressions",
            "march_ctr",
            "ctr_position_group",
            "priority_score",
            "reason_code",
            "action_label"
        ]
    ]
)

print("Number of pages selected for review:", len(top_10_review))

,client_hash_id,content_hash_id,position_bucket,march_avg_position,march_impressions,march_ctr,ctr_position_group,priority_score,reason_code,action_label
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1–3,2.693038,134984.0,0.000007,Below position median,11.772838,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,1–3,0.308426,124075.0,0.000008,Below position median,11.685355,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
2,client_23a62021009f63c4,content_44f34c0a90047651,1–3,0.665877,212404.0,0.000113,Below position median,11.631466,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1–3,0.116003,83834.0,0.000012,Below position median,11.274672,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
4,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,>6–10,7.831807,89332.0,0.000045,Below position median,11.097423,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
5,client_73cda7b4e4f265ea,content_425715547c6a3ea8,>6–10,6.983444,71513.0,0.000042,Below position median,10.899587,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
6,client_a80fca3f171ed1de,content_046fc480045b88f5,>6–10,7.208276,83788.0,0.000072,Below position median,10.854679,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
7,client_a80fca3f171ed1de,content_f0703fc6ae385591,>6–10,8.707520,63201.0,0.000032,Below position median,10.846655,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
8,client_62f4a7e64f5e0096,content_f6116743b00afc2d,>6–10,9.735658,107584.0,0.000139,Below position median,10.628108,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
9,client_62f4a7e64f5e0096,content_0c5606abaaab3178,>3–6,4.804554,38865.0,0.000000,Below position median,10.567875,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET


Number of pages selected for review: 10


### Top-10 Manual Review

The following ten pages were selected based on their highest baseline priority scores. Each recommendation is a preliminary review suggestion based on March 2026 search performance, not a confirmed diagnosis of a content problem.

| Rank | Action | Why it is there | What would make it wrong |
|---|---|---|---|
| 1 | Review the search snippet, including the title and meta description. | The page has 134,984 impressions, an average position of 2.69, and a CTR below its position-group median. Its priority score is 11.77. | Search features, query composition, or low click demand may explain the CTR rather than a defective snippet. |
| 2 | Review the search snippet for relevance and clarity. | The page records 124,075 impressions and an average position of 0.31, yet its CTR is below the position-group median. | The page may already have an effective snippet, while search intent or SERP features limit clicks. |
| 3 | Investigate the page's search presentation and relevance to user queries. | It has the highest impression volume among the ten pages, at 212,404, and its CTR is below the median for its position group. | Its high visibility may come from broad or poorly targeted queries, making snippet changes insufficient. |
| 4 | Review the title and description against the search intent the page serves. | The page has 83,834 impressions, an average position of 0.12, and below-median CTR. | The low CTR may reflect query mismatch, changing search demand, or other SERP characteristics. |
| 5 | Examine the search snippet and its relevance to the page's target queries. | With 89,332 impressions and an average position of 7.83, the page has below-median CTR for its position group. | Its ranking distribution or query mix may explain the CTR without a snippet problem. |
| 6 | Review the title and description for clarity and alignment with search intent. | The page receives 71,513 impressions, averages position 6.98, and falls below the position-group CTR median. | The page may attract impressions from queries for which its content is not the most relevant result. |
| 7 | Investigate whether the snippet communicates the page's value clearly. | It records 83,788 impressions, an average position of 7.21, and below-median CTR. | Device

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
# Inspect five lowest-priority pages selected for action
weak_picks = action_queue.tail(5).copy()

display(
    weak_picks[
        [
            "client_hash_id",
            "content_hash_id",
            "march_avg_position",
            "march_impressions",
            "march_ctr",
            "priority_score",
            "reason_code",
            "action_label"
        ]
    ]
)

print("Weak picks inspected:", len(weak_picks))

,client_hash_id,content_hash_id,march_avg_position,march_impressions,march_ctr,priority_score,reason_code,action_label
31145,client_73cda7b4e4f265ea,content_fac91559c0084f2f,2.535734,1833.0,0.002182,0.004099,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
31146,client_73cda7b4e4f265ea,content_2959297008b4b786,4.993407,455.0,0.002198,0.003588,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
31147,client_73cda7b4e4f265ea,content_419c9a021b83bbab,2.817302,33903.0,0.002183,0.003384,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
31148,client_73cda7b4e4f265ea,content_a92d40e0b4bf4347,4.378033,11827.0,0.002198,0.003119,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET
31149,client_73cda7b4e4f265ea,content_2d1f59d409a326f1,2.223974,34812.0,0.002183,0.001202,LOW_CTR_FOR_POSITION,REVIEW_SEARCH_SNIPPET


Weak picks inspected: 5


### Weak Picks Review

The five lowest-scoring pages in the action queue were inspected to identify potential weaknesses in the baseline.

| Page | Why the baseline selected it | What could make the recommendation wrong |
|---|---|---|
| 1 | The page has 1,833 impressions, an average position of 2.54, and a small positive CTR shortfall relative to its position group. | Its relatively low impression volume means the potential visibility impact may be limited. The CTR difference may also be too small to justify intervention. |
| 2 | The page has 455 impressions, an average position of 4.99, and a below-median CTR. | The low impression volume makes the estimated opportunity less significant. Its CTR may also reflect limited search demand. |
| 3 | The page has 33,903 impressions and an average position of 2.82, making it visible enough to attract attention despite its low priority score. | Its relatively small CTR shortfall may not represent a meaningful content problem. Its search queries may also explain the observed click behaviour. |
| 4 | The page records 11,827 impressions, an average position of 4.38, and a below-median CTR. | The CTR difference may be marginal, and query composition or search-result features may explain its performance. |
| 5 | The page has 34,812 impressions and an average position of 2.22, but receives a very low priority score because its relative CTR gap is small. | The page may already be performing adequately, making a snippet intervention unnecessary despite its impression volume. |

**Overall assessment:** These examples demonstrate that a positive score does not automatically justify intervention. Some pages may have only marginal CTR shortfalls, while others have limited impression volume. The baseline therefore identifies candidates for investigation rather than guaranteeing that every selected page needs improvement.

In [16]:
# Check the columns used to generate the action queue

scoring_columns = [
    "march_impressions",
    "march_ctr",
    "march_avg_position",
    "ctr_gap",
    "relative_ctr_gap",
    "priority_score",
    "reason_code",
    "action_label"
]

print("Scoring columns:")
print(scoring_columns)

# Check that future outcome fields are absent
future_columns = [
    "april_impressions",
    "april_days_observed",
    "impressions_decline_pct",
    "is_declining_label"
]

for column in future_columns:
    print(
        f"{column} present in action queue:",
        column in action_queue.columns
    )

# Confirm that the score is calculated from March information
expected_score = (
    action_queue["relative_ctr_gap"]
    * np.log1p(action_queue["march_impressions"])
)

print(
    "\nPriority score matches March-only formula:",
    np.allclose(
        action_queue["priority_score"],
        expected_score
    )
)

Scoring columns:
['march_impressions', 'march_ctr', 'march_avg_position', 'ctr_gap', 'relative_ctr_gap', 'priority_score', 'reason_code', 'action_label']
april_impressions present in action queue: True
april_days_observed present in action queue: True
impressions_decline_pct present in action queue: True
is_declining_label present in action queue: True

Priority score matches March-only formula: True


In [17]:
# Define the permitted columns for the final action queue
allowed_columns = [
    "client_hash_id",
    "content_hash_id",
    "march_impressions",
    "march_avg_position",
    "march_ctr",
    "position_bucket",
    "ctr_position_group",
    "ctr_gap",
    "relative_ctr_gap",
    "priority_score",
    "reason_code",
    "action_label"
]

# Retain only permitted columns
action_queue = action_queue[allowed_columns].copy()

# Confirm future information is absent
future_columns = [
    "april_impressions",
    "april_days_observed",
    "impressions_decline_pct",
    "is_declining_label"
]

print("Future information remaining:")

for column in future_columns:
    print(f"{column}: {column in action_queue.columns}")

print("\nFinal queue columns:")
print(action_queue.columns.tolist())

print("\nFinal queue records:", len(action_queue))

Future information remaining:
april_impressions: False
april_days_observed: False
impressions_decline_pct: False
is_declining_label: False

Final queue columns:
['client_hash_id', 'content_hash_id', 'march_impressions', 'march_avg_position', 'march_ctr', 'position_bucket', 'ctr_position_group', 'ctr_gap', 'relative_ctr_gap', 'priority_score', 'reason_code', 'action_label']

Final queue records: 31150


In [18]:
# Final leakage and integrity checks

expected_score = (
    action_queue["relative_ctr_gap"]
    * np.log1p(action_queue["march_impressions"])
)

print("Future columns excluded:",
      not any(col in action_queue.columns for col in future_columns))

print("Priority formula valid:",
      np.allclose(action_queue["priority_score"], expected_score))

print("All scores positive:",
      (action_queue["priority_score"] > 0).all())

print("Ranking descending:",
      action_queue["priority_score"].is_monotonic_decreasing)

print("Record count preserved:",
      len(action_queue) == 31150)

Future columns excluded: True
Priority formula valid: True
All scores positive: True
Ranking descending: True
Record count preserved: True


In [19]:
action_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Clean action queue exported successfully!")

Clean action queue exported successfully!


In [20]:
action_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Updated CSV:", output_path)
print("Exported records:", len(action_queue))
print("Exported columns:", len(action_queue.columns))

Updated CSV: work/outputs/baseline_action_score.csv
Exported records: 31150
Exported columns: 12


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Baseline Self-Check

**1. What problem does the baseline address?**

The baseline prioritises content pages that may benefit from search-snippet review. It identifies pages with below-median CTR relative to pages occupying similar average search positions.

**2. What information does the scoring rule use?**

The rule uses March 2026 search impressions, click-through rate, and average search position. It calculates the relative CTR gap within each position group and combines that gap with search impressions to produce a priority score.

**3. What information is excluded from scoring?**

April 2026 performance measurements, impression-decline percentages, and the decline label are excluded from the scoring inputs. April information is reserved for evaluating the baseline.

**4. What did the signal checks reveal?**

The search-volume signal produced a MIXED verdict because decline rates did not increase consistently across volume buckets. The CTR-relative-to-position signal produced a CONFIRMED verdict because below-median CTR groups consistently recorded higher subsequent decline rates across comparable position groups where a comparison was available.

**5. What are the main limitations?**

The baseline uses aggregated page-level measurements and does not account for individual search queries, search-result features, seasonality, content quality, or changes in search demand. A below-median CTR does not necessarily indicate a defective snippet. The scoring rule is therefore a prioritisation heuristic, not a causal diagnosis.

**6. How could the baseline be improved?**

Future versions could incorporate query-level performance, search intent, page freshness, content quality, and additional historical periods. These improvements should be tested using a consistent evaluation framework to determine whether they improve prioritisation.

**7. What should happen after a page is selected?**

The recommendation should trigger human review of the page, its search queries, title, meta description, and search-result presentation. Changes should only be made after the underlying problem has been investigated.